# EasyOCR

In [ ]:
import easyocr
import cv2
import math
from ultralytics import YOLO 
import csv
import time
import os


model = YOLO("./runs/detect/train/weights/best.pt")
classes = {0:"licenseplate", 1:"car"}
reader = easyocr.Reader(['en'], gpu=True) 
capture_video = cv2.VideoCapture("video7.mp4")
tiempos_inferencia_yolo = []
tiempos_inferencia_easyocr = []
frame_width = int(capture_video.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(capture_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = capture_video.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_video = cv2.VideoWriter('output_video_easyocr.mp4', fourcc, fps, (frame_width, frame_height))
csv_filename = "deteccion_de_matricula_easyocr.csv"
csv_header = [
    "fotograma", "tipo_objeto", "confianza_deteccion", "identificador_tracking", 
    "x1", "y1", "x2", "y2", "matrícula_detectada", 
    "x1_matrícula", "y1_matrícula", "x2_matrícula", "y2_matrícula", "texto_matricula_ocr",
    "tiempo_inferencia_yolo", "tiempo_inferencia_easyocr"  
]
with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    csv_writer = csv.writer(csvfile, delimiter=';')
    csv_writer.writerow(csv_header)
    
    frame_count = 0
    
    while(True):
        ret, frame_video = capture_video.read()
        if not ret:
            break
        frame_count += 1
        start_time_yolo = time.perf_counter()
        results = model.track(frame_video, persist=True,conf=0.7, tracker='bytetrack.yaml', stream=True)
        end_time_yolo = time.perf_counter()
        tiempo_inferencia_yolo = end_time_yolo - start_time_yolo
        tiempos_inferencia_yolo.append(tiempo_inferencia_yolo)
        for frames in results:
            boxes = frames.boxes
            frame_detections = []
            for box in boxes:
                cls = int(box.cls[0])
                if cls not in classes.keys():
                    continue
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                confidence = math.ceil((box.conf[0]*100))/100
                track_id = int(box.id[0]) if box.id is not None and box.id.numel() > 0 else -1
                escala = int((cls / len(classes)) * 255 * 3)
                if escala >= 255 * 2:
                    R, G, B = 255, 255, escala - 255 * 2
                elif escala >= 255:
                    R, G, B = 255, escala - 255, 0
                else:
                    R, G, B = escala, 0, 0
                plate_text = ""
                tiempo_inferencia_easyocr = 0.0
                lp_x1, lp_y1, lp_x2, lp_y2 = "", "", "", ""
                if classes[cls] == "licenseplate":
                    lp_x1, lp_y1, lp_x2, lp_y2 = x1, y1, x2, y2
                    license_plate_img = frame_video[y1:y2, x1:x2]
                    if license_plate_img.size > 0:
                        start_time_easyocr = time.perf_counter()
                        result_ocr = reader.readtext(
                            license_plate_img, 
                            allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
                            detail=0
                        )
                        end_time_easyocr = time.perf_counter()
                        tiempo_inferencia_easyocr = end_time_easyocr - start_time_easyocr
                        tiempos_inferencia_easyocr.append(tiempo_inferencia_easyocr)
                        if result_ocr:
                            plate_text = "".join(result_ocr).replace(" ", "") 
                            print(f"Matrícula Detectada: {plate_text}")
                            cv2.putText(
                                frame_video, 
                                plate_text, 
                                (x1, y1 - 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 
                                1, 
                                (0, 255, 0),
                                2
                            )
                cv2.rectangle(frame_video, (x1, y1), (x2, y2), (R, G, B), 3)
                label = f"{classes[cls]} ID:{track_id} Conf:{confidence:.2f}"
                cv2.putText(frame_video, label, [x1, y1-10], cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                time_info = f"Inf YOLO: {tiempo_inferencia_yolo:.3f}s"
                if tiempo_inferencia_easyocr > 0:
                    time_info += f" | EasyOCR: {tiempo_inferencia_easyocr:.3f}s"
                cv2.putText(frame_video, time_info, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(frame_video, f"Frame: {frame_count}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                if tiempos_inferencia_yolo:
                    avg_yolo = sum(tiempos_inferencia_yolo) / len(tiempos_inferencia_yolo)
                    avg_easyocr = sum(tiempos_inferencia_easyocr) / len(tiempos_inferencia_easyocr) if tiempos_inferencia_easyocr else 0
                    stats_text = f"Avg - YOLO: {avg_yolo:.3f}s | EasyOCR: {avg_easyocr:.3f}s"
                    cv2.putText(frame_video, stats_text, (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                row_data = [
                    frame_count,
                    classes[cls],
                    confidence,
                    track_id,
                    x1, y1, x2, y2,
                    "SÍ" if classes[cls] == "licenseplate" else "NO",
                    lp_x1, lp_y1, lp_x2, lp_y2,
                    plate_text,
                    f"{tiempo_inferencia_yolo:.6f}",      
                    f"{tiempo_inferencia_easyocr:.6f}"      
                ]
                frame_detections.append(row_data)
            csv_writer.writerows(frame_detections)
            out_video.write(frame_video)
            cv2.imshow('Deteccion y Tracking', frame_video)
        if cv2.waitKey(20) == 27:
            break

capture_video.release()
out_video.release()
cv2.destroyAllWindows()


print(f"\nProcesamiento completado!")
print(f"✓ Video guardado como: output_video.mp4")
print(f"✓ CSV guardado como: {csv_filename}")

# Tesseract

In [ ]:
import pytesseract
from pytesseract import Output
import cv2
import math
from ultralytics import YOLO 
from collections import defaultdict
import numpy as np 
import csv 
import os 

MODELO_PATH = "./runs/detect/train/weights/best.pt"
VID_PATH = "video9.mp4"

tesseract_cmd = r'C:/Program Files/Tesseract-OCR/tesseract' 
try:
    pytesseract.pytesseract.tesseract_cmd = tesseract_cmd
except Exception as e:
    print(f"Advertencia: Tesseract no se pudo configurar. El OCR no funcionará. Error: {e}")

model1 = YOLO("yolo11n.pt")  
model2 = YOLO(MODELO_PATH)   


classes_model1 = {2: "Coche", 5: "Coche", 7: "Coche"}  
classes_model2 = {0: "Matricula"}  

all_classes = ["Coche", "Matricula"]
CLASSES_TO_TRACK_MODEL1 = list(classes_model1.keys())
CLASSES_TO_TRACK_MODEL2 = list(classes_model2.keys())
COLOR_TEXTO_CONTEO = (0, 255, 255) 

capture_video = cv2.VideoCapture(VID_PATH)
if not capture_video.isOpened():
    print(f"Error: No se puede abrir el video en {VID_PATH}")
    exit()


contador_clases_unicas = {nombre: 0 for nombre in all_classes}
tracker_ids_contados_coches = set()  
tracker_ids_contados_matriculas = set() 
track_history_coches = defaultdict(lambda: [])  
track_history_matriculas = defaultdict(lambda: [])  
frame_count = 0 


frame_width = int(capture_video.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(capture_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = capture_video.get(cv2.CAP_PROP_FPS)

output_video_filename = "output_conteo.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
out_video = cv2.VideoWriter(
    output_video_filename, 
    fourcc, 
    fps, 
    (frame_width, frame_height)
)
print(f"Configurado VideoWriter para guardar en '{output_video_filename}'")

csv_filename = "conteo_y_matriculas.csv"
csv_header = [
    "fotograma", "tipo_objeto", "confianza_deteccion", "identificador_tracking", 
    "x1", "y1", "x2", "y2", "conteo_acumulado_unico_clase",
    "es_matricula", "texto_matricula_ocr"
]

with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    csv_writer = csv.writer(csvfile, delimiter=';') 
    csv_writer.writerow(csv_header)
    
    while True: 
        ret, img = capture_video.read()
        if not ret: 
            break
            
        frame_count += 1
        frame_detections = []
        
        results_model1 = model1.track(
            img, 
            persist=True, 
            conf=0.5,
            classes=CLASSES_TO_TRACK_MODEL1, 
            tracker="bytetrack.yaml", 
            verbose=False
        )
        
        if results_model1 and results_model1[0].boxes.id is not None:
            boxes_data = results_model1[0].boxes 
            
            for box in boxes_data:
                cls = int(box.cls[0])
                conf = box.conf[0].item()
                track_id = int(box.id[0].item())
                
                x1, y1, x2, y2 = [int(val) for val in box.xyxy[0].tolist()]
                
                if cls in classes_model1.keys():
                    clase_nombre = classes_model1[cls]
                    
                    if track_id not in tracker_ids_contados_coches:
                        contador_clases_unicas["Coche"] += 1
                        tracker_ids_contados_coches.add(track_id)
                    
                    color = (255, 0, 0)  
                    
                    
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                    etiqueta = f"ID:{track_id} {clase_nombre} ({conf*100:.0f}%)"
                    cv2.putText(img, etiqueta, [x1, y1 - 10], cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
                
                    center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
                    track = track_history_coches[track_id]
                    track.append((float(center_x), float(center_y)))
                    if len(track) > 30: 
                        track.pop(0)
                    if len(track) > 1:  
                        points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                        cv2.polylines(img, [points], isClosed=False, color=(230, 230, 230), thickness=3)
                    
                   
                    row_data = [
                        frame_count, 
                        clase_nombre, 
                        round(conf, 4), 
                        track_id, 
                        x1, y1, x2, y2, 
                        contador_clases_unicas["Coche"],
                        "NO",  
                        ""    
                    ]
                    frame_detections.append(row_data)
        
        
        results_model2 = model2.track(
            img, 
            persist=True, 
            classes=CLASSES_TO_TRACK_MODEL2, 
            tracker="bytetrack.yaml", 
            verbose=False
        )
        
        if results_model2 and results_model2[0].boxes.id is not None:
            boxes_data = results_model2[0].boxes 
            
            for box in boxes_data:
                cls = int(box.cls[0])
                conf = box.conf[0].item()
                track_id = int(box.id[0].item())
                
                x1, y1, x2, y2 = [int(val) for val in box.xyxy[0].tolist()]
                
                if cls in classes_model2.keys():
                    clase_nombre = classes_model2[cls]
                    
                    if track_id not in tracker_ids_contados_matriculas:
                        contador_clases_unicas["Matricula"] += 1
                        tracker_ids_contados_matriculas.add(track_id)
                    
                    color = (0, 0, 255)  
                    
                    plate_text = ""
                    es_matricula_csv = "SÍ"
                    license_plate_img = img[max(0, y1):min(frame_height, y2), max(0, x1):min(frame_width, x2)]
                    if license_plate_img.size > 0:
                            ocr_config = r'--psm 8 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
                            try:
                              
                                lp_gray = cv2.cvtColor(license_plate_img, cv2.COLOR_BGR2GRAY)
                                _, lp_thresh = cv2.threshold(lp_gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
                                
                                plate_text_raw = pytesseract.image_to_string(lp_thresh, config=ocr_config)
                                plate_text = plate_text_raw.strip().replace(" ", "").replace("\n", "").replace("\r", "")
                            except Exception:
                                plate_text = "OCR_Error_o_NoTesseract"

                        
                    if plate_text and plate_text not in ["OCR_Error_o_NoTesseract"]:
                            cv2.putText(img, plate_text, (x1, y1 - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    
                  
                    
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                    etiqueta = f"ID:{track_id} {clase_nombre} ({conf*100:.0f}%)"
                    cv2.putText(img, etiqueta, [x1, y1 - 10], cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
                    
                    
                    center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
                    track = track_history_matriculas[track_id]
                    track.append((float(center_x), float(center_y)))
                    if len(track) > 30: 
                        track.pop(0)
                    if len(track) > 1:  
                        points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                        cv2.polylines(img, [points], isClosed=False, color=(200, 200, 100), thickness=2)
                    
                    
                    row_data = [
                        frame_count, 
                        clase_nombre, 
                        round(conf, 4), 
                        track_id, 
                        x1, y1, x2, y2, 
                        contador_clases_unicas["Matricula"],
                        es_matricula_csv,
                        plate_text
                    ]
                    frame_detections.append(row_data)
            
        
        csv_writer.writerows(frame_detections)

        
        y_offset = 30
        cv2.putText(img, "--- CONTEO UNICO ACUMULADO ---", (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        y_offset += 30
        
        for nombre, cantidad in contador_clases_unicas.items():
            texto_conteo = f"Total Unicos {nombre}: {cantidad}"
            cv2.putText(img, texto_conteo, (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.8, COLOR_TEXTO_CONTEO, 2, cv2.LINE_AA) 
            y_offset += 30

        
        info_text = f"Coches trackeados: {len(tracker_ids_contados_coches)} | Matriculas trackeadas: {len(tracker_ids_contados_matriculas)}"
        cv2.putText(img, info_text, (10, frame_height - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)

    
        out_video.write(img) 
        cv2.imshow('Deteccion, Conteo y OCR', img)
    
        if cv2.waitKey(1) == 27: 
            break 
    

capture_video.release()
out_video.release()
cv2.destroyAllWindows()

print("\n--- RESUMEN FINAL DE CONTEO DE OBJETOS ÚNICOS ---")
for nombre, cantidad in contador_clases_unicas.items():
    print(f"Total acumulado de objetos únicos ({nombre}) detectados: {cantidad}")
print(f"Coches únicos: {len(tracker_ids_contados_coches)}")
print(f"Matrículas únicas: {len(tracker_ids_contados_matriculas)}")
print(f"Procesamiento finalizado. Video de salida guardado en '{output_video_filename}'.")
print(f"Datos guardados en '{csv_filename}'.")

Configurado VideoWriter para guardar en 'output_conteo.mp4'

--- RESUMEN FINAL DE CONTEO DE OBJETOS ÚNICOS ---
Total acumulado de objetos únicos (Coche) detectados: 16
Total acumulado de objetos únicos (Matricula) detectados: 18
Coches únicos: 16
Matrículas únicas: 18
Procesamiento finalizado. Video de salida guardado en 'output_conteo.mp4'.
Datos guardados en 'conteo_y_matriculas.csv'.
